[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/12-data-annotation-and-rlhf/notebooks/04-evals-rubric-and-judge.ipynb)

# 04 - Evals: rubrics, LLM-as-judge and position bias

**Goal:** build a small evaluation the way AI teams do: write a **rubric**, grade answers with a **judge**, check the judge for **position bias**, and compare the judge with human labels using kappa.

**Words you need**
- **Eval** - a repeatable test of model quality.
- **Rubric** - a scoring guide with named criteria and clear levels.
- **LLM-as-judge** - using a strong language model to grade answers.
- **Position bias** - a judge preferring whichever answer is shown first.

If `ANTHROPIC_API_KEY` is set, the judge cells call Claude. Otherwise they use a clearly-labelled **offline stand-in judge** (simple rules), so the notebook always runs.

## Step 1 - Imports and key check

We never write keys in code. `os.getenv` reads the key from the environment if it exists.

In [1]:
import os, json, re
import pandas as pd
from sklearn.metrics import cohen_kappa_score
USE_CLAUDE = bool(os.getenv("ANTHROPIC_API_KEY"))
print("Using real Claude judge" if USE_CLAUDE else "No API key -> using the OFFLINE stand-in judge")

No API key -> using the OFFLINE stand-in judge


## Step 2 - A small eval set

Each row has a question, a **reference answer** (what a correct answer must contain), and a model answer to grade. In real projects this set is written and checked by humans.

In [2]:
evalset = pd.DataFrame([
    {"q": "What is 15% of 200?", "ref": "30", "answer": "15% of 200 is 30."},
    {"q": "Name the largest planet.", "ref": "Jupiter", "answer": "Saturn is the largest planet."},
    {"q": "Give one use of a Python list.", "ref": "store", "answer": "A list can store several items in order, like names."},
    {"q": "What does CPU stand for?", "ref": "central processing unit", "answer": "Central Processing Unit."},
    {"q": "Explain recursion in one sentence.", "ref": "calls itself", "answer": "Recursion is a fancy word. It is used a lot by programmers and it is very important to learn about it in depth because it is important."},
    {"q": "Boiling point of water at sea level in Celsius?", "ref": "100", "answer": "It boils at 100 degrees Celsius."},
])
evalset

,q,ref,answer
0,What is 15% of 200?,30,15% of 200 is 30.
1,Name the largest planet.,Jupiter,Saturn is the largest planet.
2,Give one use of a Python list.,store,"A list can store several items in order, like ..."
3,What does CPU stand for?,central processing unit,Central Processing Unit.
4,Explain recursion in one sentence.,calls itself,Recursion is a fancy word. It is used a lot by...
5,Boiling point of water at sea level in Celsius?,100,It boils at 100 degrees Celsius.


## Step 3 - The rubric

Three binary (yes/no) criteria. Binary checks give higher agreement between graders than 1-10 scales. The rule for combining them is written down too.

In [3]:
RUBRIC = {
    "correct": "The answer contains the key fact from the reference answer and states nothing false.",
    "answers_question": "The answer directly addresses what was asked.",
    "concise": "The answer has no padding or repetition (roughly 25 words or fewer for these questions).",
}
COMBINE = "PASS only if 'correct' and 'answers_question' are both yes. 'concise' is reported but does not fail."
print(json.dumps(RUBRIC, indent=2)); print(COMBINE)

{
  "correct": "The answer contains the key fact from the reference answer and states nothing false.",
  "answers_question": "The answer directly addresses what was asked.",
  "concise": "The answer has no padding or repetition (roughly 25 words or fewer for these questions)."
}
PASS only if 'correct' and 'answers_question' are both yes. 'concise' is reported but does not fail.


## Step 4 - The judge

`claude_judge` shows how a real LLM-as-judge call looks: send the rubric, question, reference and answer, and ask for JSON. `offline_judge` is the stand-in: crude string rules that imitate a grader. **The offline judge is not intelligent** - it exists only so the notebook runs without a key. The model name is an example; check Anthropic's docs for current model names.

In [4]:
def offline_judge(q, ref, answer):
    """OFFLINE STAND-IN: keyword rules, not a real LLM."""
    a = answer.lower()
    correct = ref.lower() in a
    answers_question = len(a.split()) >= 2
    concise = len(a.split()) <= 25 and len(set(a.split())) / max(1, len(a.split())) > 0.7
    return {"correct": correct, "answers_question": answers_question, "concise": concise}

def claude_judge(q, ref, answer):
    import anthropic                                   # pip install anthropic
    client = anthropic.Anthropic()                     # reads ANTHROPIC_API_KEY from the environment
    prompt = (f"Grade the answer with this rubric. Reply with JSON only, keys {list(RUBRIC)} and true/false values.\n"
              f"Rubric: {json.dumps(RUBRIC)}\nQuestion: {q}\nReference: {ref}\nAnswer: {answer}")
    msg = client.messages.create(model="claude-sonnet-4-5", max_tokens=200,   # example model name
                                 messages=[{"role": "user", "content": prompt}])
    return json.loads(re.search(r"\{.*\}", msg.content[0].text, re.S).group(0))

judge = claude_judge if USE_CLAUDE else offline_judge
grades = pd.DataFrame([judge(r.q, r.ref, r.answer) for r in evalset.itertuples()])
grades["PASS"] = grades["correct"] & grades["answers_question"]
pd.concat([evalset[["q", "answer"]], grades], axis=1)

,q,answer,correct,answers_question,concise,PASS
0,What is 15% of 200?,15% of 200 is 30.,True,True,True,True
1,Name the largest planet.,Saturn is the largest planet.,False,True,True,False
2,Give one use of a Python list.,"A list can store several items in order, like ...",True,True,True,True
3,What does CPU stand for?,Central Processing Unit.,True,True,True,True
4,Explain recursion in one sentence.,Recursion is a fancy word. It is used a lot by...,False,True,False,False
5,Boiling point of water at sea level in Celsius?,It boils at 100 degrees Celsius.,True,True,True,True


## Step 5 - Compare the judge with humans

Before trusting a judge, compare it with human grades on the same items. Here a human grader marked PASS/FAIL. We compute percent agreement and Cohen's kappa (from notebook 01).

In [5]:
human_pass = [True, False, True, True, False, True]
judge_pass = list(grades["PASS"])
agree = sum(h == j for h, j in zip(human_pass, judge_pass)) / len(human_pass)
print("percent agreement:", round(agree, 2))
print("kappa            :", round(cohen_kappa_score(human_pass, judge_pass), 2))

percent agreement: 1.0
kappa            : 1.0


## Step 6 - Pairwise comparison and position bias

Often we ask a judge "which of two answers is better?". A biased judge favours position 1. The fix: ask twice with the order swapped and only count a win when both orders agree.

Our stand-in judge below is *deliberately* biased (it slightly favours whichever answer comes first) so you can see the problem and the fix.

In [6]:
def biased_pairwise_judge(q, first, second):
    """OFFLINE STAND-IN with a built-in position bias. Returns 'first' or 'second'."""
    s1 = len(set(first.lower().split())) + 3      # +3 bonus just for being shown first (the bias)
    s2 = len(set(second.lower().split()))
    return "first" if s1 >= s2 else "second"

pairs = [
    ("Explain a variable.", "A variable is a named box that stores a value.", "A variable stores a value in memory for later use."),
    ("What is an API?", "A way for programs to talk to each other.", "An API is a set of rules that lets one program request data or actions from another."),
    ("Define overfitting.", "When a model memorises training data and fails on new data.", "Overfitting: model too fit."),
]
rows = []
for q, x, y in pairs:
    ab = biased_pairwise_judge(q, x, y)          # x shown first
    ba = biased_pairwise_judge(q, y, x)          # y shown first
    winner_ab = "x" if ab == "first" else "y"
    winner_ba = "y" if ba == "first" else "x"
    final = winner_ab if winner_ab == winner_ba else "tie (inconsistent)"
    rows.append({"question": q, "x first ->": winner_ab, "y first ->": winner_ba, "final": final})
pd.DataFrame(rows)

,question,x first ->,y first ->,final
0,Explain a variable.,x,y,tie (inconsistent)
1,What is an API?,y,y,y
2,Define overfitting.,x,x,x


## Step 7 - Win rate

A **win rate** is the fraction of comparisons a model wins (ties count as half). It is the standard summary for pairwise evals.

In [7]:
final = [r["final"] for r in rows]
win_x = sum(1 for f in final if f == "x") + 0.5 * sum(1 for f in final if f.startswith("tie"))
print(f"model x win rate: {win_x / len(final):.2f}")

model x win rate: 0.50


## What you learned

- A rubric with binary, observable criteria makes grading consistent.
- An LLM judge must be checked against human labels (kappa) before you trust it.
- Swap the order in pairwise comparisons to cancel position bias.
- These are the same skills AI trainer platforms test in qualification tasks.